# Your first CUDA kernels

This notebook goes with chapters 2.1 (first CUDA program) and 9.2 (Python on the GPU).
You will add two long lists of numbers on the GPU, first in Python with Numba, then in CUDA C with nvcc.

How to run it: open it in Google Colab, choose Runtime, Change runtime type, T4 GPU, then run the cells from top to bottom.
No GPU at hand? Set the environment variable NUMBA_ENABLE_CUDASIM=1 before the first import. Numba then runs the same
kernels on the CPU in a slow simulator, which is enough to learn the ideas. The CUDA C part needs a real GPU.

## 1. Check that a GPU is there

In [ ]:
!nvidia-smi

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>!nvidia-smi</code> | Runs the NVIDIA tool in the shell (the ! sends the line to the shell). It prints the GPU name, its memory and the driver version. If it fails, the runtime has no GPU. |

In [ ]:
import os
import math
import time
import numpy as np
from numba import cuda

SIM = os.environ.get("NUMBA_ENABLE_CUDASIM") == "1"
N = 1_000 if SIM else 10_000_000
print("simulator" if SIM else cuda.get_current_device().name, "| N =", N)

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>import os</code> | os lets us read environment variables, which tell us if the simulator is on. |
| 2 | <code>import math</code> | math gives ceil, which we use to work out how many blocks we need. |
| 3 | <code>import time</code> | time lets us measure how long things take on the CPU. |
| 4 | <code>import numpy as np</code> | NumPy holds the arrays on the CPU side. Numba works directly with NumPy arrays. |
| 5 | <code>from numba import cuda</code> | cuda is the part of Numba that compiles Python functions into GPU kernels. |
| 7 | <code>SIM = os.environ.get(&quot;NUMBA_ENABLE_CUDASIM&quot;) == &quot;1&quot;</code> | True when the simulator is on. The simulator is slow, so we use a small problem there. |
| 8 | <code>N = 1_000 if SIM else 10_000_000</code> | Ten million numbers on a real GPU, one thousand in the simulator. |
| 9 | <code>print(&quot;simulator&quot; if SIM else cuda.get_current_device().name, &quot;&#124; N =&quot;, N)</code> | Prints which device we are on, so you know what the timings below mean. |

## 2. The kernel

A kernel is a function that every GPU thread runs. Each thread works out its own global index and handles one element.
This is the same formula as in chapter 2.2: global index = blockIdx.x × blockDim.x + threadIdx.x.

In [ ]:
@cuda.jit
def vec_add(a, b, c):
    i = cuda.grid(1)
    if i < c.size:
        c[i] = a[i] + b[i]

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>@cuda.jit</code> | Tells Numba to compile the next function for the GPU. It becomes a kernel, like __global__ in CUDA C. |
| 2 | <code>def vec_add(a, b, c):</code> | The kernel takes three arrays. A kernel never returns a value, it writes its result into an array. |
| 3 | <code>i = cuda.grid(1)</code> | cuda.grid(1) is the global index of this thread, blockIdx.x * blockDim.x + threadIdx.x, in one call. |
| 4 | <code>if i &lt; c.size:</code> | The bounds check. We launch whole blocks, so some threads in the last block have no element. They must do nothing. |
| 5 | <code>c[i] = a[i] + b[i]</code> | The real work: one addition per thread. Thread i touches only element i, so no two threads clash. |

## 3. Copy, launch, copy back

In [ ]:
a = np.random.rand(N).astype(np.float32)
b = np.random.rand(N).astype(np.float32)
d_a = cuda.to_device(a)
d_b = cuda.to_device(b)
d_c = cuda.device_array_like(a)

threads = 256
blocks = math.ceil(N / threads)
vec_add[blocks, threads](d_a, d_b, d_c)
c = d_c.copy_to_host()

print("blocks:", blocks, "threads per block:", threads, "total threads:", blocks * threads)
print("correct:", np.allclose(c, a + b))

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>a = np.random.rand(N).astype(np.float32)</code> | N random numbers between 0 and 1. float32 because GPUs are fastest in 32 bit floats. |
| 2 | <code>b = np.random.rand(N).astype(np.float32)</code> | A second input list of the same size. |
| 3 | <code>d_a = cuda.to_device(a)</code> | Copies a from CPU memory to GPU memory (cudaMalloc plus cudaMemcpy host to device in CUDA C). |
| 4 | <code>d_b = cuda.to_device(b)</code> | Copies b to the GPU the same way. |
| 5 | <code>d_c = cuda.device_array_like(a)</code> | Reserves GPU memory for the result with the same shape and type as a. Nothing is copied, the kernel fills it. |
| 7 | <code>threads = 256</code> | 256 threads per block, a multiple of the warp size 32 and a common safe choice. |
| 8 | <code>blocks = math.ceil(N / threads)</code> | Enough blocks to cover N. Ceil rounds up, so the last block may be partly empty, which the bounds check handles. |
| 9 | <code>vec_add[blocks, threads](d_a, d_b, d_c)</code> | The launch. [blocks, threads] is Numba's way of writing <<<blocks, threads>>>. |
| 10 | <code>c = d_c.copy_to_host()</code> | Copies the result back to the CPU. This call also waits for the kernel to finish. |
| 12 | <code>print(&quot;blocks:&quot;, blocks, &quot;threads per block:&quot;, threads, &quot;total threads:&quot;, blocks * threads)</code> | Shows the launch shape. Total threads is a little more than N because of rounding up. |
| 13 | <code>print(&quot;correct:&quot;, np.allclose(c, a + b))</code> | Compares with NumPy on the CPU. allclose allows tiny rounding differences. |

## 4. Time it honestly

The first launch also compiles the kernel, so we warm up once. A kernel launch returns at once, so we call
cuda.synchronize() before we stop the clock. Otherwise we would time only the launch, not the work.

In [ ]:
vec_add[blocks, threads](d_a, d_b, d_c)
cuda.synchronize()
t0 = time.perf_counter()
for _ in range(10):
    vec_add[blocks, threads](d_a, d_b, d_c)
cuda.synchronize()
gpu_ms = (time.perf_counter() - t0) / 10 * 1000

t0 = time.perf_counter()
c_cpu = a + b
cpu_ms = (time.perf_counter() - t0) * 1000

gb = 3 * N * 4 / 1e9
print(f"GPU kernel {gpu_ms:.3f} ms, about {gb / (gpu_ms / 1000):.0f} GB/s")
print(f"NumPy on CPU {cpu_ms:.3f} ms")

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>vec_add[blocks, threads](d_a, d_b, d_c)</code> | Warm up launch. The first call compiles the kernel, which we do not want to time. |
| 2 | <code>cuda.synchronize()</code> | Waits until the GPU is idle, so the clock starts from a clean state. |
| 3 | <code>t0 = time.perf_counter()</code> | Start time, from a clock with sub microsecond precision. |
| 4 | <code>for _ in range(10):</code> | Ten launches, so one slow run does not decide the result. |
| 5 | <code>vec_add[blocks, threads](d_a, d_b, d_c)</code> | The same launch as before. Launches return before the GPU finishes. |
| 6 | <code>cuda.synchronize()</code> | Waits for all ten kernels. Without it we would time only the launches. |
| 7 | <code>gpu_ms = (time.perf_counter() - t0) / 10 * 1000</code> | Average time per launch in milliseconds. |
| 9 | <code>t0 = time.perf_counter()</code> | Start time for the CPU version. |
| 10 | <code>c_cpu = a + b</code> | NumPy adds the two lists on the CPU. |
| 11 | <code>cpu_ms = (time.perf_counter() - t0) * 1000</code> | CPU time in milliseconds. |
| 13 | <code>gb = 3 * N * 4 / 1e9</code> | Bytes moved per launch: read a and b, write c, 4 bytes each, in gigabytes. |
| 14 | <code>print(f&quot;GPU kernel {gpu_ms:.3f} ms, about {gb / (gpu_ms / 1000):.0f} GB/s&quot;)</code> | Effective bandwidth = bytes ÷ time. Vector add does almost no maths, so this number is what limits it (chapter 4.2). |
| 15 | <code>print(f&quot;NumPy on CPU {cpu_ms:.3f} ms&quot;)</code> | The CPU time, for comparison. Note it does not include copies to the GPU, which we did once at the start. |

## 5. The grid stride loop

A grid stride loop lets a fixed number of threads cover any N. Each thread starts at its index and jumps by the total
number of threads in the grid. This is the pattern from chapter 2.2.

In [ ]:
@cuda.jit
def vec_add_stride(a, b, c):
    start = cuda.grid(1)
    stride = cuda.gridsize(1)
    for i in range(start, c.size, stride):
        c[i] = a[i] + b[i]

vec_add_stride[32, 128](d_a, d_b, d_c)
print("stride version correct:", np.allclose(d_c.copy_to_host(), a + b))

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>@cuda.jit</code> | Compile the next function as a kernel. |
| 2 | <code>def vec_add_stride(a, b, c):</code> | Same inputs and output as before. |
| 3 | <code>start = cuda.grid(1)</code> | This thread's global index, where it starts. |
| 4 | <code>stride = cuda.gridsize(1)</code> | gridsize(1) is blocks × threads per block, the number of threads in the whole grid. |
| 5 | <code>for i in range(start, c.size, stride):</code> | Visit start, start + stride, start + 2 × stride and so on until the end. The loop condition is the bounds check. |
| 6 | <code>c[i] = a[i] + b[i]</code> | Same work as before, now done several times per thread. |
| 8 | <code>vec_add_stride[32, 128](d_a, d_b, d_c)</code> | Only 32 × 128 = 4096 threads, far fewer than N, and the loop still covers everything. |
| 9 | <code>print(&quot;stride version correct:&quot;, np.allclose(d_c.copy_to_host(), a + b))</code> | Check the result against NumPy. |

## 6. The same program in CUDA C

The cell below writes a file with %%writefile. The next cell compiles it with nvcc and runs it.
CUDA_CHECK stops the program with a clear message when a CUDA call fails (chapter 9.1).

In [ ]:
%%writefile vec_add.cu
#include <cstdio>
#include <cstdlib>
#include <cuda_runtime.h>

#define CUDA_CHECK(call) do { cudaError_t e = (call); if (e != cudaSuccess) { \
    printf("CUDA error %s at %s:%d\n", cudaGetErrorString(e), __FILE__, __LINE__); exit(1); } } while (0)

__global__ void vecAdd(const float* a, const float* b, float* c, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < n) c[i] = a[i] + b[i];
}

int main() {
    const int n = 1 << 20;
    size_t bytes = n * sizeof(float);
    float *a = (float*)malloc(bytes), *b = (float*)malloc(bytes), *c = (float*)malloc(bytes);
    for (int i = 0; i < n; i++) { a[i] = 1.0f; b[i] = 2.0f; }

    float *da, *db, *dc;
    CUDA_CHECK(cudaMalloc(&da, bytes));
    CUDA_CHECK(cudaMalloc(&db, bytes));
    CUDA_CHECK(cudaMalloc(&dc, bytes));
    CUDA_CHECK(cudaMemcpy(da, a, bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(db, b, bytes, cudaMemcpyHostToDevice));

    int threads = 256, blocks = (n + threads - 1) / threads;
    vecAdd<<<blocks, threads>>>(da, db, dc, n);
    CUDA_CHECK(cudaGetLastError());
    CUDA_CHECK(cudaMemcpy(c, dc, bytes, cudaMemcpyDeviceToHost));

    printf("c[0] = %.1f, c[n-1] = %.1f\n", c[0], c[n - 1]);
    cudaFree(da); cudaFree(db); cudaFree(dc);
    free(a); free(b); free(c);
    return 0;
}

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>%%writefile vec_add.cu</code> | A notebook command that saves the rest of the cell to the file vec_add.cu instead of running it as Python. |
| 2 | <code>#include &lt;cstdio&gt;</code> | printf, for printing from the host. |
| 3 | <code>#include &lt;cstdlib&gt;</code> | malloc, free and exit. |
| 4 | <code>#include &lt;cuda_runtime.h&gt;</code> | The CUDA runtime API: cudaMalloc, cudaMemcpy and friends. nvcc adds it on its own, we write it to be clear. |
| 6 | <code>#define CUDA_CHECK(call) do { cudaError_t e = (call); if (e != cudaSuccess) { \</code> | A macro that runs a CUDA call and, if it did not return cudaSuccess, prints the error with the file and line. |
| 7 | <code>printf(&quot;CUDA error %s at %s:%d\n&quot;, cudaGetErrorString(e), __FILE__, __LINE__); exit(1); } } while (0)</code> | The second half of the macro. do { } while (0) makes the macro behave like one statement after an if. |
| 9 | <code>__global__ void vecAdd(const float* a, const float* b, float* c, int n) {</code> | __global__ marks a kernel: it runs on the GPU and is launched from the CPU. const says the inputs are only read. |
| 10 | <code>int i = blockIdx.x * blockDim.x + threadIdx.x;</code> | The global index of this thread, the same formula that cuda.grid(1) computes in Numba. |
| 11 | <code>if (i &lt; n) c[i] = a[i] + b[i];</code> | Bounds check and the addition, one element per thread. |
| 12 | <code>}</code> | End of the kernel. |
| 14 | <code>int main() {</code> | The host program starts here. |
| 15 | <code>const int n = 1 &lt;&lt; 20;</code> | n = 2 to the power 20 = 1,048,576 elements. |
| 16 | <code>size_t bytes = n * sizeof(float);</code> | The size of one array in bytes. size_t is used for sizes because it can hold large values. |
| 17 | <code>float *a = (float*)malloc(bytes), *b = (float*)malloc(bytes), *c = (float*)malloc(bytes);</code> | Host (CPU) arrays for the two inputs and the output. |
| 18 | <code>for (int i = 0; i &lt; n; i++) { a[i] = 1.0f; b[i] = 2.0f; }</code> | Fill the inputs, so every answer should be 3.0. |
| 20 | <code>float *da, *db, *dc;</code> | Pointers that will point into GPU memory. |
| 21 | <code>CUDA_CHECK(cudaMalloc(&amp;da, bytes));</code> | Reserve GPU memory for a. We pass the address of the pointer so cudaMalloc can fill it in. |
| 22 | <code>CUDA_CHECK(cudaMalloc(&amp;db, bytes));</code> | Reserve GPU memory for b. |
| 23 | <code>CUDA_CHECK(cudaMalloc(&amp;dc, bytes));</code> | Reserve GPU memory for the result. |
| 24 | <code>CUDA_CHECK(cudaMemcpy(da, a, bytes, cudaMemcpyHostToDevice));</code> | Copy a from the CPU to the GPU. |
| 25 | <code>CUDA_CHECK(cudaMemcpy(db, b, bytes, cudaMemcpyHostToDevice));</code> | Copy b from the CPU to the GPU. |
| 27 | <code>int threads = 256, blocks = (n + threads - 1) / threads;</code> | 256 threads per block and the rounded up number of blocks. (n + 255) / 256 is ceil in integer maths. |
| 28 | <code>vecAdd&lt;&lt;&lt;blocks, threads&gt;&gt;&gt;(da, db, dc, n);</code> | The launch: blocks × threads copies of vecAdd run on the GPU. |
| 29 | <code>CUDA_CHECK(cudaGetLastError());</code> | A launch returns no error code, so we ask for the last error. This catches a bad launch shape. |
| 30 | <code>CUDA_CHECK(cudaMemcpy(c, dc, bytes, cudaMemcpyDeviceToHost));</code> | Copy the result back. This call waits for the kernel to finish first. |
| 32 | <code>printf(&quot;c[0] = %.1f, c[n-1] = %.1f\n&quot;, c[0], c[n - 1]);</code> | Print the first and last element. Both should be 3.0. |
| 33 | <code>cudaFree(da); cudaFree(db); cudaFree(dc);</code> | Give the GPU memory back. |
| 34 | <code>free(a); free(b); free(c);</code> | Give the CPU memory back. |
| 35 | <code>return 0;</code> | Return 0 to the shell to say all went well. |
| 36 | <code>}</code> | End of main. |

In [ ]:
!nvcc -O2 -arch=sm_75 -o vec_add vec_add.cu
!./vec_add

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>!nvcc -O2 -arch=sm_75 -o vec_add vec_add.cu</code> | Compile with nvcc. -arch=sm_75 builds for the T4 GPU in Colab (compute capability 7.5). Use sm_80 for an A100. |
| 2 | <code>!./vec_add</code> | Run the program. It should print c[0] = 3.0, c[n-1] = 3.0. |

## 7. Try these

1. Remove the bounds check in the Numba kernel and set N to a value that is not a multiple of 256. On a real GPU run it under compute-sanitizer to see the out of bounds write (chapter 9.1).
2. Change threads to 32, 64, 128, 512 and 1024 and time each. Explain the result with occupancy (chapter 2.4).
3. Write a kernel for c = 2a + b (SAXPY). Work out its bytes and its FLOPs, then its arithmetic intensity.